# Reproduce the paper figures

This notebook runs the validated calculations in `pre_figures.py` and exports the six main figures plus the supplemental integration-region schematic. It follows the revised manuscript dated 27 September 2026.

**Prerequisite:** the Figure 3 archive must exist at `data/pre_figure3_10000`. A fresh clone needs the simulation and analysis commands in [the reproduction guide](../docs/reproducibility.md). Generated data are not committed to Git.

The seven figure-specific notebooks remain as source/provenance records. This notebook is the entry point for the revised figures: it calls the public package with its default adaptive integration and regenerates every theoretical curve/grid. It preserves the verified 50,000-trial Figure 3 ensemble and its confidence intervals.


In [1]:
from pathlib import Path
import json
import os
import sys

here = Path.cwd().resolve()
CODE = next(p for p in (here, *here.parents) if (p / "gaussian_crossings").is_dir())
OUTPUT = Path(os.environ.get("GAUSSIAN_CROSSINGS_ARCHIVE", CODE / "data/pre_figures_adaptive"))
FIGURES = Path(os.environ.get("GAUSSIAN_CROSSINGS_FIGURES", CODE / "figures/publication"))
os.environ["MPLCONFIGDIR"] = str(OUTPUT / "build/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE))
sys.path.insert(0, str(CODE / "examples"))
import pre_figures as pre
print("Archive:", OUTPUT)
print("Figures:", FIGURES)


Archive: /Users/shivangrawat/python_projects/gaussian-crossings/build/publication-audit/recalculated
Figures: /Users/shivangrawat/python_projects/gaussian-crossings/build/publication-audit/figures


## Calculate or verify a cached run

For a fresh calculation, choose a new `OUTPUT` directory. Existing numerical arrays are never overwritten. Cached arrays are accepted only when their archived source is intact and the numerical runner and recorded package source hashes agree with the current code; rendering and command-line changes are allowed. Independent numerical validation is rerun before export.


In [2]:
if (OUTPUT / "arrays.npz").exists():
    pre.verify_archive(OUTPUT)
    print(json.dumps(pre.validate(), indent=2))
    print("Reusing verified numerical arrays.")
else:
    pre.calculate(OUTPUT)
print((OUTPUT / "validation.json").read_text())


{
  "rq_small_lag_covariance_max_relative_error": 3.6459502084085216e-10,
  "sdho_matrix_exponential_max_abs_error": 7.549516567451064e-15,
  "positive_pair_integral_max_abs_error": 1.6653345369377348e-16,
  "current_package_integrand_max_abs_error": 1.1563666690861396e-14,
  "ou_rescaling_max_abs_error": 1.27675647831893e-15,
  "positive_full_integral_max_abs_error": 1.9539925233402755e-14,
  "fig3_recomputed_curve_max_abs_change": 3.552713678800501e-15,
  "fig3_counts_hashes_and_statistics_verified": true,
  "fig3_reused_trials_per_damping_ratio": 10000
}
Reusing verified numerical arrays.
{
  "rq_small_lag_covariance_max_relative_error": 3.6459502084085216e-10,
  "sdho_matrix_exponential_max_abs_error": 7.549516567451064e-15,
  "positive_pair_integral_max_abs_error": 1.6653345369377348e-16,
  "current_package_integrand_max_abs_error": 1.1563666690861396e-14,
  "ou_rescaling_max_abs_error": 1.27675647831893e-15,
  "positive_full_integral_max_abs_error": 1.9539925233402755e-14,
  "fig

In [3]:
pre.plot(OUTPUT, FIGURES)
print((OUTPUT / "comparisons.json").read_text())
print("Exported figures:")
for path in sorted(FIGURES.glob("*.pdf")):
    print(" -", path.name)


Exported 7 figures to /Users/shivangrawat/python_projects/gaussian-crossings/build/publication-audit/figures
{
  "sdho_peaks": [
    {
      "zeta": 0.5,
      "a_peak": 3.225640264408681,
      "f_peak": 1.0289556472578307,
      "first_F1_crossing": 2.439698717248535,
      "f_at_zero": 0.33851065601368857,
      "minimum_on_0_20": 0.33851065601368857
    },
    {
      "zeta": 1.0,
      "a_peak": 2.1699569733235853,
      "f_peak": 1.2226364009937634,
      "first_F1_crossing": 1.1434995320574723,
      "f_at_zero": 0.592232303449979,
      "minimum_on_0_20": 0.592232303449979
    },
    {
      "zeta": 2.0,
      "a_peak": 1.7920388943100045,
      "f_peak": 1.9943403025529944,
      "first_F1_crossing": null,
      "f_at_zero": 1.0600817663064719,
      "minimum_on_0_20": 1.0147005449107747
    }
  ],
  "rq_peaks": [
    {
      "alpha": "0.75",
      "a_peak": 1.7841768583444229,
      "f_peak": 1.3358545148405023,
      "first_F1_crossing": 0.9399121215718719,
      "f_at_zero"

## Interpretation and numerical checks

- The mean rate is determined by the covariance at zero lag and its curvature. Variance and Fano factors use the full lag dependence.
- At sufficiently high thresholds, even the plotted underdamped oscillator can have a Fano factor above one.
- No super-to-sub-Poissonian return is resolved within the plotted OU parameter range.
- All five rational-quadratic curves, including the squared-exponential limit, have a maximum above one.
- Figure 3 uses finite-window predictions at T = 120 and the complete archived ensemble. Its confidence intervals are pointwise; two marginal interval misses are retained.

The optional historical manuscript-comparison PDF is available through the separate `comparison --paper ...` command. It is not needed to regenerate the paper figures.
